# The Vibe Tax — the extraction-pitfall catalogue, with reproducible examples

A **glass-box, executable** version of `EXTRACTION_PITFALLS.md`. Each failure mode is shown on a
**real model completion** (loaded live from `lcb_v3_responses.json`) where possible, or a **minimal,
clearly-labeled synthetic reproduction** where the mode does not occur in this particular
capable-model LiveCodeBench set. For every mode we show: the trigger, the **naive** extractor
failing, and **our** extractor recovering — so a reviewer can rerun it, not just read a claim.

**Why this matters for the paper.** Rows 1–3 and 10 are *prose/selection* errors whose trigger —
how much explanatory text the model emits — co-varies with the model and the prompt wrapper. That
is exactly why a prose-sensitive extractor turns a *null* prompt-style effect into a *significant*
one. This notebook is the evidence behind that claim.

**How to run.** In Colab: upload `lcb_v3_responses.json` when prompted (no git clone needed). In the
repo: it is read from `data/vibe_tax_lcb/lcb_v3_responses.json` automatically. No model is ever
re-queried — every example re-scores a stored completion.


## 1 · Load the stored completions

In [ ]:
import json, os, io, ast, re, inspect, contextlib

# Works both in the repo and in Colab (upload when prompted).
CANDIDATES = ["lcb_v3_responses.json",
              "data/vibe_tax_lcb/lcb_v3_responses.json",
              "../data/vibe_tax_lcb/lcb_v3_responses.json"]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:
    try:
        from google.colab import files           # Colab upload path
        print("Upload lcb_v3_responses.json:")
        up = files.upload()
        path = next(iter(up))
    except Exception:
        raise FileNotFoundError("lcb_v3_responses.json not found; upload it or run from the repo.")

RESP = json.load(open(path))
print(f"loaded {len(RESP)} records from {path}")
print("models:", sorted({r['model'] for r in RESP}),
      "| conditions:", sorted({r['level'] for r in RESP}))

def rec(task_id, model, level):
    '''Fetch one stored completion by (task_id, model, condition).'''
    return next(r for r in RESP
               if r['task_id']==task_id and r['model']==model and r['level']==level)


## 2 · The two extractors (verbatim from the three-extractor notebook)

`extract_naive` is the pre-fix pipeline: first fenced block naming the target, else the whole
fence-stripped reply **including any prose**. `extract_ours` is the robust extractor from
`score_lcb.py`: trim-to-compilable, keep helpers, shape-agnostic target resolution.
`describe(code, entry)` reports what happens when you try to *run* the extracted code.

In [ ]:
def _trim_to_compilable(code):
    '''Drop trailing lines until the source parses — defeats trailing prose.'''
    lines = code.split("\n")
    while lines:
        src = "\n".join(lines)
        try:
            ast.parse(src); return src
        except SyntaxError:
            lines.pop()
    return None

def extract_ours(completion, entry):
    if not completion: return None
    text = "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    code_blocks = [b for b in blocks if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]
    cands = []
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)
    if m: cands.append(text[m.start():])                 # keep helpers defined above Solution
    if code_blocks: cands.append("\n\n".join(code_blocks))  # helper + Solution in separate blocks
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b: cands.append(b)
    for anchor in ("class Solution", f"def {entry}"):
        i = text.find(anchor)
        if i != -1: cands.append(text[i:])
    for c in cands:
        t = _trim_to_compilable(c)
        if t and ("class Solution" in t or f"def {entry}" in t): return t
    return cands[0] if cands else None

def extract_naive(completion, entry):
    if not completion: return None
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b: return b
    if "class Solution" in completion or f"def {entry}" in completion:
        return "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    return None

def resolve_callable(ns, entry):
    '''Shape-agnostic: bare function / renamed class / nested class, preferring `Solution`.'''
    f = ns.get(entry)
    if callable(f) and not inspect.isclass(f): return f
    classes = [v for v in ns.values() if inspect.isclass(v)]
    nested = [v for c in classes for v in vars(c).values() if inspect.isclass(v)]
    cands = [c for c in (classes + nested) if callable(getattr(c, entry, None))]
    if not cands: return None
    cands.sort(key=lambda c: c.__name__ != "Solution")
    try: inst = cands[0]()
    except Exception: return None
    return getattr(inst, entry, None)

_IMPORTS = ('from typing import *\nimport collections, math, heapq, bisect, itertools, functools, re\n'
            'from collections import *\nfrom math import *\nfrom functools import *\n')

def describe(code, entry):
    '''Try to parse + exec the extracted code; report the outcome a grader would see.'''
    if code is None: return "NULL — extractor returned nothing (scored as no-code FAIL)"
    try: ast.parse(code)
    except SyntaxError as e: return f"SyntaxError at line {e.lineno}: {e.msg}  (correct code scored FAIL)"
    ns = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()):   # mute solutions that print at import
            exec(_IMPORTS + code, ns)
    except Exception as e: return f"{type(e).__name__} at exec: {e}  (e.g. missing helper -> FAIL)"
    old = callable(getattr(ns.get('Solution'), entry, None))   # what the OLD harness checked
    new = resolve_callable(ns, entry) is not None              # shape-agnostic resolution
    if new and not old: return f"OK via resolve_callable; OLD harness (Solution-only) MISSED it -> FN"
    if new: return "OK — parses, defines target, runnable"
    return "parses but target not found (FN)"

def show(task_id, model, level):
    r = rec(task_id, model, level); comp, entry = r['completion'], r['entry_point']
    print(f"[{task_id} | {model} | {level}]  entry_point = {entry}  (fences: {comp.count('```')})")
    print("  naive  ->", describe(extract_naive(comp, entry), entry))
    print("  ours   ->", describe(extract_ours(comp, entry), entry))
    return r
print("extractors + describe() ready.")


## 3 · Population view — how often each mode fires in the shipped set

Before the one-at-a-time walkthrough, the aggregate: across all 2,004 stored completions, how many
does the naive extractor mishandle while ours recovers? This is the catalogue, counted.

In [ ]:
from collections import Counter
tally = Counter()
for r in RESP:
    comp, entry = r['completion'], r['entry_point']
    n = describe(extract_naive(comp, entry), entry)
    o = describe(extract_ours(comp, entry), entry)
    our_ok = o.startswith("OK")
    if n.startswith("NULL") and our_ok:               tally["10 · null extraction"] += 1
    elif n.startswith("SyntaxError") and our_ok:      tally["1/2 · prose breaks parse"] += 1
    elif ("MISSED" in n) and our_ok:                  tally["6 · class-shape (old-harness miss)"] += 1
    elif n.startswith(("NameError","TypeError")) and our_ok: tally["5 · dropped helper"] += 1
for k,v in sorted(tally.items(), key=lambda kv:-kv[1]):
    print(f"  {v:4d}  mode {k}")
print(f"\n  total naive-mishandled / ours-recovered: {sum(tally.values())} of {len(RESP)}")


## 4 · Mode 1 — trailing prose after the code  *(real, 139 records)*

**Trigger:** the system prompt says "plain Python, no fences", and the model complies — then appends
an explanation *after* the code ("Time complexity is O(n·q)…"). **Mechanism:** with no fence to mark
where the code ends, the naive extractor feeds code **+ trailing English** to `ast.parse` →
`SyntaxError` on otherwise-correct code. The polite/detailed wrapper elicits more explain-after-code,
so this penalizes it hardest — the artifact that manufactures the "politeness tax". **Fix:**
`_trim_to_compilable` drops trailing lines until it parses.

In [ ]:
r = show('lcb/3517', 'chatgpt', 'webchat_detailed')
print("\n--- last 240 chars of the raw completion (the trailing prose) ---")
print(r['completion'][-240:])


## 5 · Mode 2 — leading prose / chatter before the code  *(real, 126 records)*

**Trigger:** the reply opens with explanation ("Looking at this problem, I need to…") before any
code, again with no fences. **Mechanism:** naive returns the whole fence-stripped reply; the leading
English is now line 1 → `SyntaxError`. **Fix:** ours anchors to the first top-level code construct
(`^(class|def|import|from|@)`) and trims.

In [ ]:
r = show('lcb/3535', 'claude', 'agentic_casual')
print("\n--- first 200 chars of the raw completion (the leading prose) ---")
print(r['completion'][:200])


## 6 · Mode 6 — class-shape mismatch  *(real, 70 records)*

**Trigger:** the model returns a **bare module-level function** `def shortestDistanceAfterQueries(...)`
instead of a method on `class Solution` (a code *smell*, not a wrong answer). **Mechanism:** the old
harness did `Solution().<entry>(*args)` and found nothing → false negative, even though the code is
correct and runnable. **Fix:** `resolve_callable` binds to the implementation regardless of shape
(bare function, renamed class, nested class). Note the `-> FN` tag on the naive line below.

In [ ]:
r = show('lcb/3517', 'codestral', 'agentic_terse')
print("\n--- first 220 chars (note: bare function, no `class Solution`) ---")
print(r['completion'][:220])


## 7 · Modes not observed in this capable-model set — minimal reproductions

Modes 5, 10 and 12 **do not fire** on the shipped ChatGPT/Claude/Codestral LCB completions (these
models reliably emit a `class Solution` with its helpers inline, so there is nothing to drop, miss,
or truncate). That absence is itself a finding — but the modes are real on other models/runs and are
documented in the catalogue, so here are **minimal, clearly-labeled synthetic** completions that
trigger each one, scored by the exact same extractors.

### 7a · Mode 5 — dropped helper defined above the class  *(synthetic)*

A helper class (`DSU`, `Fenwick`, …) defined in its **own fenced block** before the `class Solution`
block. The naive "first block naming the target" rule grabs only the Solution block, so `DSU` is
gone. This does **not** fail at parse time — `DSU` is referenced *inside* a method, so the class
still imports — it fails only when the test **calls** the method (`NameError`). We therefore run the
method to expose it. Our extractor concatenates the code blocks, keeping the helper.

In [ ]:
SYNTH_M5 = '''Here is the solution.

```python
class DSU:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x: x = self.p[x]
        return x
```

And the main class:

```python
class Solution:
    def countComponents(self, n, edges):
        d = DSU(n)
        for a, b in edges: d.p[d.find(a)] = d.find(b)
        return len({d.find(i) for i in range(n)})
```
'''
entry = 'countComponents'

def call_method(code, entry, *args):
    '''Extract-free runtime check: exec code, resolve the target, CALL it, return result or error.'''
    if code is None: return "NULL (no code)"
    try: ast.parse(code)
    except SyntaxError as e: return f"SyntaxError: {e.msg}"
    ns = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()): exec(_IMPORTS + code, ns)
    except Exception as e: return f"{type(e).__name__} at import: {e}"
    fn = resolve_callable(ns, entry)
    if fn is None: return "target not found"
    try: return f"returned {fn(*args)}"
    except Exception as e: return f"{type(e).__name__} when called: {e}"   # <-- the DSU NameError lands here

args = (5, [[0,1],[1,2],[3,4]])        # 2 components
print("naive -> extracted Solution block only; calling the method:")
print("        ", call_method(extract_naive(SYNTH_M5, entry), entry, *args))
print("ours  -> kept the helper; calling the method:")
print("        ", call_method(extract_ours(SYNTH_M5, entry), entry, *args))


### 7b · Mode 10 — over-aggressive extraction returns NULL  *(synthetic)*

A prose-heavy reply with no fences, no `class Solution`, and a solution function **named differently
from the entry point** (`kadane`, not the expected `maxSubArray`). The naive anchor check
(`"class Solution"` / `"def <entry>"`) matches nothing → it returns `None` → scored as a silent
no-code FAIL. Ours' last-resort fallback recovers the *compilable code* from the first top-level
construct, converting a silent null into a **diagnosable shape-mismatch** (the symbol is there, just
under the wrong name) rather than nothing at all.

In [ ]:
SYNTH_M10 = '''I'll solve this with a simple linear scan over the array.

def kadane(nums):
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x); best = max(best, cur)
    return best

That runs in O(n) time and O(1) space.'''
entry = 'maxSubArray'                 # expected name; the model wrote `kadane` instead
nai, our = extract_naive(SYNTH_M10, entry), extract_ours(SYNTH_M10, entry)
print("naive returned:", "None (silent no-code FAIL)" if nai is None else repr(nai[:40]))
print("ours  returned:", "None" if our is None else f"{len(our)} chars of recovered text (NOT None)")
print("  naive ->", describe(nai, entry))
print("  ours  ->", describe(our, entry))
print("\nHonest reading: with the target under a different name and no class, NEITHER extractor can")
print("bind `maxSubArray`. The only difference is naive fails SILENTLY (None), while ours surfaces")
print("the code so the failure is a DIAGNOSABLE shape/name mismatch, not an invisible no-code FAIL.")


### 7c · Mode 12 — output-token cap truncates the completion  *(synthetic)*

A larger/reasoning model spends its token budget before finishing, so the visible answer is cut off
**mid-statement** → it never parses. This is a *harness* artifact (the cap, not the model) and it
hits the **more capable** models hardest. In our diagnostics, **65 of 101** hard "failures" for
GPT-5.6 were empty or truncated completions — false fails misread as incapacity. The fix is to raise
the budget and flag empty vs parse-incomplete completions separately, not to score them as wrong.

In [ ]:
SYNTH_M12 = '''```python
class Solution:
    def maxProfit(self, prices):
        best = 0
        for i in range(len(prices)):
            for j in range(i+1, len(pri'''      # <-- cut off mid-token
entry = 'maxProfit'
print("naive ->", describe(extract_naive(SYNTH_M12, entry), entry))
print("ours  ->", describe(extract_ours(SYNTH_M12, entry), entry))
print("\nNote: no extractor can recover a truncated completion — the fix is upstream",
      "(raise MAX_TOKENS; flag empties/truncations separately), not in extraction.")


## 8 · Harness / metric modes (11, 13, 14) — need the gated test suite

Three catalogue modes live in the *grader*, not the extractor, so they need LCB's official tests
(`lcb_tests.jsonl`, HF-gated, not in the repo):

- **Mode 11 — per-problem total timeout.** A correct-but-slow solution on a many-test problem is
  killed because the *sum* of test times exceeds one budget. Fix: per-test timeout with early exit
  (`passes()` in `score_lcb.py`). Reproduce by scoring with a single whole-suite budget vs per-test.
- **Mode 13 — weak base tests pass subtly-wrong code** (false positives). Fix: re-score on
  HumanEval+ edge tests (`score_vibe_tax_plus.py`).
- **Mode 14 — binary pass@1 hides partial correctness.** Fix: test-level partial credit
  (`full_partial_credit.py`).

These are described here for completeness; run them from the scoring scripts once the gated tests
are available (`extract_lcb_tests.py` with an HF token).

## 9 · The through-line (what this notebook proves)

- **Modes 1, 2, 10** are *prose/selection* errors. Their trigger — explanatory-text volume —
  **co-varies with the prompt wrapper**, so a prose-sensitive extractor converts a null prompt-style
  effect into a significant one. This is the mechanism behind the "politeness tax".
- **Mode 6** is a *structure* error: a code smell (bare/renamed/nested target), not an incorrect
  answer, that the old harness scored FAIL.
- **Modes 5, 12** and the harness modes **11, 13, 14** don't touch extraction selection but produce
  the same symptom — a correct model scored wrong (or a wrong model scored right, mode 13).
- In every extraction case above, **our extractor recovers the same code the naive pipeline
  discarded**, and it only ever *removes* text — it cannot inflate correctness. That is why the
  register effect vanishes under robust extraction without the pass-rate being inflated.

Cross-references: `EXTRACTION_PITFALLS.md` (the prose catalogue), `THREE_EXTRACTOR_RESULTS.md` (the
aggregate result), `The_Vibe_Tax_Three_Extractors.ipynb` (naive vs ours vs EvalPlus `sanitize`).